# Aurora · Kaggle GPU worker

Turn Kaggle's free GPU into an Aurora backend (lipsync by default).

**Before running:**
1. **Settings** → Accelerator = **GPU** (T4 ×2 or P100), Internet = **ON**.
2. **Add-ons → Secrets**: `NGROK_AUTHTOKEN`, `NGROK_STATIC_DOMAIN`, `AURORA_URL`, `AURORA_REGISTER_KEY` (optional: `AURORA_WORKER_TOKEN`, `AURORA_TASKS`, `AURORA_WORKER_REPO_RAW`).
3. **Run All**.

See `workers/kaggle/README.md` for where to get each secret. The worker auto-registers in **Admin → Workers** and serves lip-sync jobs for free.

In [ ]:
# Aurora · Kaggle GPU worker — one-cell launcher.
# Settings: Accelerator = GPU (T4/P100), Internet = ON.
# Add-ons -> Secrets: NGROK_AUTHTOKEN, NGROK_STATIC_DOMAIN, AURORA_URL,
#   AURORA_REGISTER_KEY (+ optional AURORA_WORKER_TOKEN, AURORA_TASKS).
# Then Run All. This fetches the launcher from your repo and runs it.
import os, urllib.request

# For a renamed repo, a non-default branch, or a public mirror, set this secret to
# your raw base, e.g. https://raw.githubusercontent.com/OWNER/REPO/BRANCH/workers
# (a private repo won't fetch over raw URLs — upload the worker files instead).
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    try:
        _v = _us.get_secret("AURORA_WORKER_REPO_RAW")
        if _v:
            os.environ["AURORA_WORKER_REPO_RAW"] = _v.strip()
    except Exception:
        pass
except Exception:
    pass

_explicit = os.environ.get("AURORA_WORKER_REPO_RAW", "").strip().rstrip("/")
_bases = [_explicit] if _explicit else [
    f"https://raw.githubusercontent.com/josephcrown920/aurora-charm-forge-87e3e757/{b}/workers"
    for b in ("main", "Main", "master")
]
_dst, _err = "/kaggle/working/aurora_worker_kaggle.py", None
for _b in _bases:
    try:
        urllib.request.urlretrieve(f"{_b}/kaggle/aurora_worker_kaggle.py", _dst)
        break
    except Exception as e:
        _err = e
else:
    raise SystemExit(
        f"Could not fetch launcher from {_bases}: {_err}\n"
        "Set the AURORA_WORKER_REPO_RAW secret or make the repo public."
    )
exec(open(_dst).read())
